In [13]:
# 이상치 탐지
# %pwd
import pandas as pd
import numpy as np
df = pd.read_csv(r'data\messy_data.csv')
na_cols = df.isna().any()[df.isna().any().values].index
# 이상치 탐지를 위해서 결측치를 중앙값으로 대처
for col in na_cols:
    df[col].fillna(df[col].median)

In [14]:
df.shape

(112, 6)

In [15]:
df[na_cols].describe()

,age,salary,score
count,105.000000,106.000000,101.000000
mean,45.723810,59225.556604,49.884059
std,34.346228,62843.972979,28.385438
min,-5.000000,-50000.000000,0.520000
25%,31.000000,39391.000000,27.890000
50%,41.000000,50703.000000,48.560000
75%,55.000000,61663.500000,70.040000
max,300.000000,500000.000000,99.630000


In [16]:
df['age'].quantile(0.25)

np.float64(31.0)

In [17]:
# IQR 
def detect_outlier_iqr(data,column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3-Q1
    lower = Q1 - IQR*1.5
    upper = Q3 + IQR*1.5
    outlier_mask = (data[column] > upper ) | (data[column] < lower)
    return outlier_mask,lower,upper,Q1,Q3,IQR

In [18]:
numeric_cols = df.describe().columns[ 1: ]

iqr_df = df.copy()
total_removed_counts = 0
for colname in numeric_cols:    
    outlier_mask,lower,upper,Q1,Q3,IQR =  detect_outlier_iqr(iqr_df,colname)
    print(f'컬럼명 : {colname}')
    print(f'이상치 개수 : {outlier_mask.sum()}')
    print('-'*50)
    before = len(iqr_df)
    iqr_df = iqr_df[~outlier_mask]
    removed = before - len(iqr_df)
    total_removed_counts += removed

print(f'IQR 제거후 shape {iqr_df.shape}')
print(f'총 제거수 : {total_removed_counts}')

컬럼명 : age
이상치 개수 : 3
--------------------------------------------------
컬럼명 : salary
이상치 개수 : 5
--------------------------------------------------
컬럼명 : score
이상치 개수 : 0
--------------------------------------------------
IQR 제거후 shape (104, 6)
총 제거수 : 8
